## Neural Network Regression with PyTorch

### This notebook demonstrates **how a simple neural network learns a regression task** using **PyTorch**.

### We'll cover:
1. Creating synthetic data for regression  
2. Defining a neural network model  
3. Training the model using a loss function and optimizer  
4. Visualizing the learning process

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)

In [ ]:
# Generate synthetic data 
X = torch.linspace(-5, 5, 100).unsqueeze(1)     # shape [100, 1] , unsqueeze to make it 2D, unsqueeze(1) adds a dimension at index 1 which is columns
y = 3 * X + 2 + torch.randn(X.size()) * 1.5   # add noise , shape [100, 1], torch.randn generates random numbers from standard normal distribution
# y = 3*X**2 + 2*X + 10                           # Non-linear data for testing

In [ ]:
X.shape, y.shape

In [ ]:
Z = torch.linspace(-5, 5, 100)
print(f' Z : {Z.shape} , Z : {Z[::5]}, Z.dtype : {Z.dtype}')

In [ ]:
print(f' X : {X.shape}, y : {y.shape} ')
print(f' X : {X[:5]}, y : {y[:5]} ')

In [ ]:
# Visualize 
plt.scatter(X.numpy(), y.numpy(), label='Data') # Convert to numpy for plotting
plt.title("X tensor vs y tensor") 
plt.xlabel("X") 
plt.ylabel("y") 
plt.legend() 
plt.grid(True)
plt.show()

## Step 3: Define a Simple Neural Network We'll use a **feedforward neural network** with: 
- Input layer: 1 neuron (for x) 
- Hidden layer: 10 neurons (with ReLU) 
- Output layer: 1 neuron (for predicted y)

In [ ]:
# Define Neural Network Model

class RegressionNN(nn.Module): 
    def __init__(self): 
        super().__init__() 
        self.net = nn.Sequential( 
            nn.Linear(1, 20), 
            nn.ReLU(), 
            nn.Linear(20, 1) 
        ) 
    def forward(self, x): 
        return self.net(x) 

In [ ]:
# Instantiate model
model = RegressionNN() 
print(model)

In [ ]:
# Model parameters

model.state_dict()  # View model parameters, weights and biases from each layer.

In [ ]:
# View model parameters with names
for name, param in model.named_parameters(): 
    print(f'Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n') 

In [ ]:
# Model Parameters and Gradients

for param in model.parameters(): 
        print(f"Parameter shape: {param.shape}")
        if param.grad is not None:
            print(f"Parameter gradients: {param.grad}")

## Step 4: Define Loss Function and Optimizer We'll use: 
- **Mean Squared Error (MSE)** for regression loss 
- **Adam optimizer** for efficient learning 

In [ ]:
criterion = nn.MSELoss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)

## Step 5: Training the Model The training loop involves: 
1. Forward pass — compute predictions 
2. Compute loss (MSE between predictions and actual y) 
3. Backward pass — compute gradients 
4. Update parameters

In [ ]:
epochs = 500 
losses = [] 
for epoch in range(epochs): 

    # Forward pass 

    y_pred = model(X)  # X: input data, y_pred: predicted output
    loss = criterion(y_pred, y)  # Compute loss between predicted and actual output, using Mean Squared Error.

    # Backward pass and optimization 
    optimizer.zero_grad()  # Clears the gradients of all parameters tracked by that optimizer so the next backward pass starts from zero
    loss.backward() # Computes the gradient of current loss w.r.t. parameters (or anything requiring gradients)
    optimizer.step() # Updates the parameters based on the current gradients

    losses.append(loss.item()) # Store loss value for plotting

    # Print progress 
    if (epoch+1) % 50 == 0: 
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
# Check the final loss
# Get max and min loss
print(f"Max Loss: {max(losses):.4f}, Min Loss: {min(losses):.4f}")
print(f"Final Loss: {loss.item():.4f}")

## Step 6: Visualize Loss Curve

In [ ]:
# Plot loss curve

plt.plot(losses) 
plt.title("Training Loss Curve") 
plt.xlabel("Epoch") 
plt.ylabel("MSE Loss") 
plt.grid(True)
plt.show()

## Step 7: Compare Predictions vs True Data


In [ ]:
# Evaluate model 
model.eval() # Set the model to evaluation mode. NOTE: affects layers like dropout, batchnorm etc.

# No gradient calculation needed during evaluation
with torch.no_grad(): 
    predicted = model(X)

plt.scatter(X.numpy(), y.numpy(), label='True Data') 
plt.plot(X.numpy(), predicted.numpy(), color='red', label='Model Prediction') 
plt.title("Model Prediction vs True Data") 
plt.xlabel("x") 
plt.ylabel("y") 
plt.legend() 
plt.show()

In [ ]:
print(f'y_pred : {predicted[:5]}')
print(f'y_true : {y[:5]}')

# Step 8: Evaluate how well y_pred fits y

In [ ]:
with torch.no_grad():
    mse = criterion(y_pred, y).item()
    mae = torch.mean(torch.abs(y_pred - y)).item() # Mean Absolute Error: average of absolute differences between predicted and actual values

    ss_res = torch.sum((y - y_pred) ** 2) # Residual Sum of Squares
    ss_tot = torch.sum((y - torch.mean(y)) ** 2) # Total Sum of Squares
    r2 = (1 - ss_res / ss_tot).item() # R-squared: Coefficient of Determination

print(f"MSE: {mse:.4f}") # Mean Squared Error
print(f"MAE: {mae:.4f}") # Mean Absolute Error
print(f"R^2: {r2:.4f}") # R-squared: Coefficient of Determination

## Step 9: Plots to evaluate model performance

In [ ]:
# (1) y_true vs y_pred with y=x reference, 
# (2) residuals vs X and histogram
# (3) residuals histogram


y_np = y.numpy().ravel() # Convert true y to numpy array and flatten.
y_pred_np = y_pred.detach().numpy().ravel() # Detach from computation graph, convert to numpy and flatten.
X_np = X.detach().numpy().ravel()  # flatten X for plotting

residuals = (y_np - y_pred_np) # Calculate residuals, y_true - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) parity plot
axes[0].scatter(y_np, y_pred_np, s=20, alpha=0.7)
minv, maxv = min(y_np.min(), y_pred_np.min()), max(y_np.max(), y_pred_np.max())
axes[0].plot([minv, maxv], [minv, maxv], color='red', linestyle='--')
axes[0].set_xlabel("y_true")
axes[0].set_ylabel("y_pred")
axes[0].set_title("Parity plot (y_true vs y_pred)")

# 2) residuals vs X
axes[1].scatter(X_np, residuals, s=20, alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel("X")
axes[1].set_ylabel("Residual (y_true - y_pred)")
axes[1].set_title("Residuals vs X")

# 3) residuals histogram
axes[2].hist(residuals, bins=30, alpha=0.8)
axes[2].axvline(residuals.mean(), color='red', linestyle='--', label=f"mean={residuals.mean():.3f}")
axes[2].set_xlabel("Residual")
axes[2].set_title("Residuals histogram")
axes[2].legend()

plt.tight_layout()
plt.show()

## Step 10: Summary We built and trained a **simple neural network for regression** using PyTorch. 
Key takeaways: 
- Neural networks can approximate even simple functions like y = 3x + 2 and more alikes.
- The loss function guides the model to minimize prediction error.
- With each epoch, parameters update to better fit the data. 

### Try experimenting by: 
- Adding more layers or neurons. 
- Changing activation functions (e.g., `Tanh`). 
- Modifying the learning rate or noise level.